Import

In [1]:
import pandas as pd

Reading Files

In [2]:
essays = pd.read_csv('kdd-cup-2014-predicting-excitement-at-donors-choose/data/essays.csv')
outcomes = pd.read_csv('kdd-cup-2014-predicting-excitement-at-donors-choose/data/outcomes.csv')
projects = pd.read_csv('kdd-cup-2014-predicting-excitement-at-donors-choose/data/projects.csv')
donations = pd.read_csv('kdd-cup-2014-predicting-excitement-at-donors-choose/data/donations.csv')

Merging

In [3]:
combined = projects.merge(outcomes,how='inner',on='projectid').merge(essays,how='inner',on='projectid')

In [4]:
combined.isna().sum()

projectid                                              0
teacher_acctid_x                                       0
schoolid                                               0
school_ncesid                                      38848
school_latitude                                        0
school_longitude                                       0
school_city                                            0
school_state                                           0
school_zip                                             4
school_metro                                       75488
school_district                                      922
school_county                                         17
school_charter                                         0
school_magnet                                          0
school_year_round                                      0
school_nlns                                            0
school_kipp                                            0
school_charter_ready_promise   

Preparing the Data

In [5]:
#dropping columns that are not important such as teacher account id
#dropping columns with too many categories such as school city

columns = ['projectid','school_state','school_metro','school_magnet','school_nlns',
           'school_kipp','school_charter_ready_promise','teacher_teach_for_america',
           'teacher_ny_teaching_fellow','primary_focus_subject','primary_focus_area',
           'resource_type','poverty_level','grade_level','total_price_excluding_optional_support',
           'total_price_including_optional_support','students_reached','eligible_double_your_impact_match',
           'eligible_almost_home_match','date_posted','fully_funded','short_description','need_statement','essay']

data = combined[columns]

In [6]:
#convert t/f values to binary for relevant categorical data

columns_binary = ['school_magnet','school_nlns',
           'school_kipp','school_charter_ready_promise','teacher_teach_for_america',
           'teacher_ny_teaching_fellow','eligible_double_your_impact_match',
           'eligible_almost_home_match','fully_funded']

binary_map = {'f':0,'t':1}

for col in columns_binary:
    data.loc[:,col] = data[col].map(binary_map)

data

,projectid,school_state,school_metro,school_magnet,school_nlns,school_kipp,school_charter_ready_promise,teacher_teach_for_america,teacher_ny_teaching_fellow,primary_focus_subject,...,total_price_excluding_optional_support,total_price_including_optional_support,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded,short_description,need_statement,essay
0,62526d85d2a1818432d03d600969e99c,IL,suburban,0,0,0,0,0,0,Special Needs,...,444.36,522.78,7.0,0,0,2013-12-31,1,"If they can't learn the way we teach, we teach...","My students need puzzles, games and visual lea...","If they can't learn the way we teach, we teach..."
1,33d59ac771b80222ad63ef0f4ac47ade,ID,urban,0,0,0,0,0,0,Mathematics,...,233.24,274.40,30.0,0,0,2013-12-31,0,"Which is bigger, three liters or three quarts?...",My students need capacity tools and resources ...,"Which is bigger, three liters or three quarts?..."
2,1a3aaeffc56dd2a421e37d8298024c0a,NH,suburban,0,0,0,0,0,0,Environmental Science,...,285.09,335.40,230.0,0,0,2013-12-31,0,Do you remember classrooms that used just book...,My students need UV sensitive beads and a Vern...,Do you remember classrooms that used just book...
3,33aa19ee4da4c5adf47d0dfb84fab5ef,VA,urban,0,0,0,0,0,0,Literacy,...,232.94,274.05,18.0,0,0,2013-12-31,0,My class was given the beautiful gift of books...,My students need labels and book bins to organ...,My class was given the beautiful gift of books...
4,e31c0ea8b68f404699dfb0d39e9bc99b,IL,urban,1,0,0,0,0,0,Environmental Science,...,513.41,604.01,70.0,1,0,2013-12-31,1,Thinking back in school science was either rea...,My students need an Interactive whiteboard les...,Thinking back in school science was either rea...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619321,a7236ea96c812895cafc5d700d779147,NY,urban,0,0,0,0,0,0,Environmental Science,...,231.00,281.71,0.0,0,0,2002-09-17,1,My name is Mary Reide. I am a New York City T...,"The cost of this proposal is [price], includin...",My name is Mary Reide. I am a New York City T...
619322,e02da37beb332eb66c2d2ba989c597ad,NY,urban,0,0,0,0,0,0,Economics,...,1129.00,1376.83,0.0,0,0,2002-09-17,1,I teach economics to 25 students at Satellite ...,"The cost of this proposal is $1377, including ...",I teach economics to 25 students at Satellite ...
619323,82e536f14eadf2671a70e03416f695a3,NY,urban,1,0,0,0,0,0,Early Development,...,125.00,152.44,0.0,0,0,2002-09-16,1,I just returned from 3 weeks in S. Africa. I w...,"The cost of this proposal is [price], includin...",I just returned from 3 weeks in S. Africa. I ...
619324,e139df754a873a62d93daa56acbf8040,NY,NaN,0,0,0,0,0,0,Literacy,...,125.00,152.44,0.0,0,0,2002-09-13,1,I teach 9th grade Humanities at Vanguard High ...,"The cost of this proposal is [price], includin...",I teach 9th grade Humanities at Vanguard High ...


In [8]:
#dropping outliers for requested funds

q1 = data['total_price_excluding_optional_support'].quantile(0.25)
q3 = data['total_price_excluding_optional_support'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 0.75 * iqr
upper_bound = q3 + 5.5 * iqr

# checking how much is being droppped 
# (data['total_price_excluding_optional_support'] > upper_bound).sum()
# (data['total_price_excluding_optional_support'] < lower_bound).sum()

data = data[(data['total_price_excluding_optional_support'] >= lower_bound)]
data = data[(data['total_price_excluding_optional_support'] <= upper_bound)]

print(
data['total_price_excluding_optional_support'].describe(),'\n',
data['total_price_excluding_optional_support'].skew() #decreased from before cleaning
)

count    611585.000000
mean        465.091509
std         295.040331
min          37.920000
25%         265.370000
50%         407.470000
75%         567.590000
max        2240.700000
Name: total_price_excluding_optional_support, dtype: float64 
 2.1624981055055006


In [20]:
#dropping outliers for students

q1 = data['students_reached'].quantile(0.25)
q3 = data['students_reached'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 0.05 * iqr
upper_bound = q3 + 21.5 * iqr

print(lower_bound,upper_bound)

# (data['total_price_excluding_optional_support'] < lower_bound).sum()
# (data['total_price_excluding_optional_support'] > upper_bound).sum()

data = data[(data['students_reached'] >= lower_bound)]
data = data[(data['students_reached'] <= upper_bound)]

print(
data['students_reached'].describe(),'\n',
data['students_reached'].skew() #decreased from before cleaning
)

18.3 1687.0
count    535159.000000
mean        100.414172
std         153.508650
min          19.000000
25%          25.000000
50%          40.000000
75%         100.000000
max        1663.000000
Name: students_reached, dtype: float64 
 3.4576687444892755


In [21]:
#sort ascending chronologically

data = data.sort_values(by='date_posted')

#drop missing value

percent_loss = round((data.shape[0] - data.dropna().shape[0])/data.shape[0],4)

print(f'{percent_loss*100}%')

12.22%


In [23]:
data.dropna(inplace=True)
data

,projectid,school_state,school_metro,school_magnet,school_nlns,school_kipp,school_charter_ready_promise,teacher_teach_for_america,teacher_ny_teaching_fellow,primary_focus_subject,...,total_price_excluding_optional_support,total_price_including_optional_support,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded,short_description,need_statement,essay
618176,e05428ddfb30e295844feb11b9d33553,NY,urban,0,0,0,0,0,0,Other,...,574.00,700.00,20.0,0,0,2003-06-19,1,"I am a first grade teacher at PS 7, Abraham Li...","The cost of a Language Center, purchased at ww...","I am a first grade teacher at PS 7, Abraham Li..."
617736,16605cd564a9611768636b5d0925cb9f,NY,urban,0,0,0,0,0,1,Social Sciences,...,258.30,315.00,45.0,0,0,2003-07-09,1,"I am a third grade teacher at PS 169, a title ...",The cost of 28 book titles about the continent...,"I am a third grade teacher at PS 169, a title ..."
617455,b3d9d45512362d48ba2b701ac7b87d11,NY,urban,0,0,0,0,0,0,Character Education,...,236.88,288.88,25.0,0,0,2003-08-16,1,I am a second grade teacher in a Title I schoo...,The cost for a supply of prizes such as pencil...,I am a second grade teacher in a Title I schoo...
617364,67d25016a34607c3284f05e343bca777,NY,urban,0,0,0,0,0,0,Literacy,...,234.62,286.12,20.0,0,0,2003-09-01,1,Hello. I am a third grade teacher at P.S.169. ...,"The cost of a Rotating Wire Book Rack is $287,...",Hello. I am a third grade teacher at P.S.1...
617325,3710a8c738487f7eafc2c903a2f46cb2,NY,urban,0,0,0,0,0,0,Literacy,...,917.89,1119.38,32.0,0,0,2003-09-08,1,"I am a music teacher, turned third grade teach...","The cost of creating a library, complete with ...","I am a music teacher, turned third grade teach..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,f820ef3537f4445b0716244fae36f763,ID,urban,0,0,0,0,0,0,Mathematics,...,277.37,326.32,30.0,0,0,2013-12-31,0,Math and reading are two separate subjects in ...,My students need a large range of math stories...,Math and reading are two separate subjects in ...
131,95ee208a51831edffa7cc2e0aa3e83cd,AZ,urban,0,0,0,0,0,0,Literacy,...,202.34,238.05,98.0,0,0,2013-12-31,1,Remember the first book you ever loved? How yo...,My students need books to improve their readin...,Remember the first book you ever loved? How yo...
130,383500f017fea60562ba737b051e1d21,MN,suburban,0,0,0,0,0,0,Literature & Writing,...,537.50,632.35,29.0,0,0,2013-12-31,1,"I will be honest with you, I'm horrible at pre...",My students need funding for a workshop at The...,"I will be honest with you, I'm horrible at pre..."
139,b16bef1607c79b0974eff9ba3db58da2,IL,urban,0,0,0,0,0,0,Mathematics,...,174.56,205.36,35.0,1,0,2013-12-31,1,Every morning and afternoon my students and I ...,My students need a pencil sharpener and pencil...,Every morning and afternoon my students and I ...


EDA

In [24]:
data[['total_price_excluding_optional_support','students_reached','fully_funded']].groupby('fully_funded').describe()

total_price_excluding_optional_support                          \
                                              count        mean         std   
fully_funded                                                                  
0                                          138720.0  572.529243  331.602523   
1                                          331017.0  430.800017  274.709238   

                                                     students_reached  \
                min    25%      50%     75%      max            count   
fully_funded                                                            
0             38.00  361.0  475.925  725.74  2240.67         138720.0   
1             37.92  241.9  378.150  510.77  2240.70         331017.0   

                                                                       
                    mean         std   min   25%   50%    75%     max  
fully_funded                                                           
0             105.787197  162.181586  19.0  25.0  40.0  110.0  1663.0  
1              98.393279  150.723566  19.0  25.0  40.0  100.0  1600.0